In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages,MessagesState
from dotenv import load_dotenv 
from langchain_groq import ChatGroq
from IPython.display import Image,display
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
import os 


In [ ]:
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="qwen/qwen3.6-27b")


In [ ]:

# 1. Define your tool using the @tool decorator
@tool
def get_weather(city: str) -> str:
    """Returns the weather for a given city."""
    return f"The weather in {city} is 72°F and sunny."

tools = [get_weather]




In [ ]:

# 2. Bind the tools to the LLM
# (Assuming 'llm' is your initialized ChatModel like ChatGroq or ChatOpenAI)
llm_with_tools = llm.bind_tools(tools)


In [ ]:
# 3. Create the Agent Node
def agent_node(state: MessagesState):
    # The LLM reads the messages and decides whether to answer or call a tool
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


In [ ]:
# 4. Create the Tool Execution Node
# ToolNode automatically executes the requested function and returns a ToolMessage
tool_node = ToolNode(tools)

In [ ]:



# 5. Build and Wire the Graph
workflow = StateGraph(MessagesState)

workflow.add_node("agent", agent_node)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")

In [ ]:

# 6. The Router
# tools_condition checks the LLM's response. 
# If it sees a tool call -> routes to "tools".
# If it sees a normal text response -> routes to END.
workflow.add_conditional_edges(
    "agent",
    tools_condition,
)

# Once the tool finishes executing, it must loop back to the agent
workflow.add_edge("tools", "agent")

app = workflow.compile()